# 18 — Magnitude Sweep Sensitivity Analysis

This notebook holds the fixed representative portfolio from `mc_representative_portfolio.json` constant and varies one intervention's magnitude at a time across 0%, 25%, 50%, 75%, 100%, 150%, and 200% of its nominal magnitude. It records the static study-area aggregate E/Ec/S and composite scores.

This is a static end-state comparison, not a lifecycle or temporal run: every grid point starts from `initialize_state()` and does not call `advance_year()`. Coefficient values remain nominal; only intervention magnitudes vary.

## Engine and coupling choices

The notebook imports the existing engine without editing it. Each action is applied with `apply_action(state, action_id, location, magnitude, _skip_coupling=False, climate_context=None)`: coupling remains ON so coupled effects contribute to realistic scoring, while `climate_context=None` uses the historical/default context. `initialize_state()` uses defaults, including `baseline_retirements=None`, because this is a single-year snapshot rather than an `advance_year()` run.

A 0% point omits the target action entirely. For positive points, the complete portfolio is applied in the fixed JSON order, with only the target magnitude replaced. Engine-enforced `ValueError` constraints are logged and skipped; unexpected exception types are allowed to stop execution.

In [1]:
import copy
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = DATA_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import terra_engine as te

PORTFOLIO_PATH = DATA_DIR / 'mc_representative_portfolio.json'
LIBRARY_PATH = DATA_DIR / 'mw_action_library_v3.json'
VALIDATION_PATH = DATA_DIR / 'mc_validation_priority.csv'
with PORTFOLIO_PATH.open() as f:
    PORTFOLIO = json.load(f)['portfolio']
with LIBRARY_PATH.open() as f:
    LIBRARY = json.load(f)
ACTIONS = LIBRARY['actions']
VALIDATION_DF = pd.read_csv(VALIDATION_PATH)
GRID_PCTS = [0, 25, 50, 75, 100, 150, 200]
CAPITALS = ['E', 'Ec', 'S']

BASE_STATE = te.initialize_state()
assert set(entry['action_id'] for entry in PORTFOLIO).issubset(BASE_STATE['action_library']['actions'])
print(f'Engine module: {te.__file__}')
print(f'Fixed portfolio actions: {len(PORTFOLIO)}')
def raw_study_area_scores(state):
    counties = state.get('county_ees', {})
    if not counties:
        raise ValueError('initialize_state returned no county_ees for raw aggregate')
    weighted = {capital: 0.0 for capital in CAPITALS}
    total_population = 0.0
    for ees in counties.values():
        population = float(ees.get('population', 1.0))
        total_population += population
        for capital in CAPITALS:
            weighted[capital] += float(ees[capital]) * population
    if total_population <= 0:
        raise ValueError('county_ees population total must be positive')
    weighted = {capital: weighted[capital] / total_population for capital in CAPITALS}
    weighted['composite_score'] = float(np.mean([weighted[capital] for capital in CAPITALS]))
    return weighted

rounded_initial = te.compute_ees_summary(BASE_STATE)['study_area']
raw_initial = raw_study_area_scores(BASE_STATE)
print(f'Initial rounded study-area E/Ec/S: {rounded_initial}')
print(f'Initial raw study-area E/Ec/S/composite: {raw_initial}')

Engine module: /Users/dylanhartman/projects/Energy Modeling/energy-map/src/terra_engine.py
Fixed portfolio actions: 30
Initial rounded study-area E/Ec/S: {'E': 3.1098, 'Ec': 6.9322, 'S': 5.2827, 'E_baseline': 3.1098, 'Ec_baseline': 6.9322, 'S_baseline': 5.2827}
Initial raw study-area E/Ec/S/composite: {'E': 3.109765349642827, 'Ec': 6.932164204057357, 'S': 5.282735347213421, 'composite_score': 5.108221633637869}


## Cost normalization

The library does not consistently expose `capex_per_unit`. The notebook now admits a cost basis to the sourced ranking only when it has a real citation and a unit compatible with the action magnitude; annual maintenance costs, unitless costs, and `$ / job` versus facility mismatches are excluded. The raw slope is `Δcomposite / Δmagnitude`; `cost_efficiency_per_$1M` converts that slope to composite points per $1M of the selected sourced per-unit cost. Internal slope math uses unrounded raw state aggregates; rounded `compute_ees_summary` values are display checks.

In [2]:
def _number(value):
    return isinstance(value, (int, float, np.integer, np.floating)) and not isinstance(value, bool) and np.isfinite(value)

def _unit_compatible(cost_unit, magnitude_unit):
    cost_unit = str(cost_unit or '').lower()
    cost_unit = cost_unit.split('(')[0].strip()
    magnitude_unit = str(magnitude_unit or '').lower()
    if not cost_unit or '/year' in cost_unit:
        return False
    aliases = (
        ('acre', 'acre'), ('household', 'household'), ('worker', 'worker'),
        ('clinic', 'facilit'), ('facility', 'facilit'), ('unit', 'unit'),
        ('mile', 'mile'), ('mw', 'mw'), ('station', 'station'),
        ('connection', 'connection'), ('watershed', 'watershed'),
    )
    return any(left in cost_unit and right in magnitude_unit for left, right in aliases)

def capex_per_unit_info(action):
    explicit = action.get('capex_per_unit')
    explicit_source = action.get('capex_source') or action.get('cost_source') or action.get('atb_source')
    magnitude_unit = action.get('unit_label') or action.get('unit')
    if _number(explicit) and explicit > 0 and explicit_source:
        return float(explicit), str(explicit_source), True, 'explicit capex_per_unit'
    for year in ('2025', '2023', '2035', '2050'):
        atb = action.get(f'atb_capex_{year}')
        atb_source = action.get('atb_source')
        if _number(atb) and atb > 0 and atb_source:
            unit = str(action.get('atb_capex_unit', ''))
            if '$/kW' in unit and 'mw' in str(magnitude_unit).lower():
                return float(atb) * 1000.0, str(atb_source), True, f'atb_capex_{year} converted $/kW to $/MW'
    cost_source = action.get('cost_source')
    cost_unit = action.get('cost_unit')
    if cost_source and _unit_compatible(cost_unit, magnitude_unit):
        cost = action.get('cost_2024')
        if _number(cost) and cost > 0:
            return float(cost), str(cost_source), True, 'cost_2024 with matching sourced unit'
        low = action.get('cost_low_2024')
        high = action.get('cost_high_2024')
        if _number(low) and _number(high) and low > 0 and high > 0:
            return float((low + high) / 2), str(cost_source), True, 'midpoint of sourced cost_low_2024/cost_high_2024'
    if cost_source and cost_unit:
        reason = f'cost unit {cost_unit!r} does not match magnitude unit {magnitude_unit!r}'
    elif cost_source:
        reason = 'sourced cost has no usable unit'
    else:
        reason = 'no capex/cost source citation'
    return np.nan, '', False, reason

CAPEX_META = {}
for entry in PORTFOLIO:
    value, source, usable, reason = capex_per_unit_info(ACTIONS[entry['action_id']])
    CAPEX_META[entry['action_id']] = {
        'capex_per_unit': value, 'capex_source': source,
        'usable_sourced_capex': usable, 'capex_reason': reason,
    }

cost_reference = pd.DataFrame([
    {'action_id': e['action_id'], 'capital_tier': ACTIONS[e['action_id']].get('bucket'), **CAPEX_META[e['action_id']]}
    for e in PORTFOLIO
])
display(cost_reference)

,action_id,capital_tier,capex_per_unit,capex_source,usable_sourced_capex,capex_reason
0,wind_utility,energy_generation,1430000.0,"NREL ATB 2024 v3.0 — Land-Based Wind, Moderate...",True,atb_capex_2025 converted $/kW to $/MW
1,solar_utility,energy_generation,1555200.0,ATB 2024 local file — UtilityPV,True,atb_capex_2023 converted $/kW to $/MW
2,geothermal_utility,energy_generation,NaN,,False,no capex/cost source citation
3,coal_repowering,energy_generation,300000.0,NETL Cost and Performance Baseline for Fossil ...,True,cost_2024 with matching sourced unit
4,smr_advanced,energy_generation,NaN,,False,no capex/cost source citation
5,coal_to_solar,energy_generation,NaN,,False,no capex/cost source citation
6,battery_grid,energy_storage,NaN,,False,no capex/cost source citation
7,transmission_230kv,energy_transmission,2500000.0,"Liming & Tegen (2011, NREL TP-5500-48175); DOE...",True,cost_2024 with matching sourced unit
8,microgrid,energy_transmission,NaN,,False,no capex/cost source citation
9,riparian_buffer,hydrological_restoration,8000.0,USDA EQIP Practice 391 (Riparian Forest Buffer...,True,cost_2024 with matching sourced unit


In [3]:
class SweepConstraintError(ValueError):
    pass

def run_scenario(target_action_id, magnitude_pct):
    target = next(entry for entry in PORTFOLIO if entry['action_id'] == target_action_id)
    target_magnitude = target['magnitude'] * magnitude_pct / 100.0
    state = te.initialize_state()
    skipped_actions = []
    for entry in PORTFOLIO:
        action_id = entry['action_id']
        magnitude = target_magnitude if action_id == target_action_id else entry['magnitude']
        if magnitude <= 0:
            continue
        action_meta = ACTIONS[action_id]
        if entry['geoid'] not in action_meta.get('applicable_counties', []):
            raise SweepConstraintError(
                f'{action_id} at {entry["geoid"]} violates applicable_counties'
            )
        try:
            state, _ = te.apply_action(
                state, action_id, entry['geoid'], magnitude,
                _skip_coupling=False, climate_context=None
            )
        except ValueError as exc:
            reason = f'{action_id} at {entry["geoid"]}, magnitude={magnitude}: {exc}'
            if action_id == target_action_id:
                raise SweepConstraintError(reason) from exc
            skipped_actions.append({'action_id': action_id, 'reason': reason})
    # Keep the public rounded summary for the engine contract/display check,
    # but use the raw county_ees values for internal sweep math.
    rounded_study = te.compute_ees_summary(state)['study_area']
    scores = raw_study_area_scores(state)
    assert all(abs(rounded_study[capital] - round(scores[capital], 4)) < 1e-12 for capital in CAPITALS)
    return scores, skipped_actions

point_rows = []
constraint_log = []
for entry in PORTFOLIO:
    action_id = entry['action_id']
    for magnitude_pct in GRID_PCTS:
        magnitude = entry['magnitude'] * magnitude_pct / 100.0
        try:
            scores, skipped_actions = run_scenario(action_id, magnitude_pct)
            point_status = 'valid' if not skipped_actions else 'valid_with_skipped_action'
            point_reason = '; '.join(item['reason'] for item in skipped_actions)
            for item in skipped_actions:
                constraint_log.append({
                    'action_id': action_id, 'magnitude_pct': magnitude_pct,
                    'magnitude': magnitude, 'reason': item['reason'],
                    'scope': 'fixed portfolio action omitted; target point retained'
                })
        except SweepConstraintError as exc:
            reason = str(exc)
            constraint_log.append({
                'action_id': action_id, 'magnitude_pct': magnitude_pct,
                'magnitude': magnitude, 'reason': reason
            })
            point_rows.append({
                'action_id': action_id, 'capital_tier': ACTIONS[action_id].get('bucket'),
                'magnitude_pct': magnitude_pct, 'magnitude': magnitude,
                'status': 'skipped', 'reason': reason,
                **{capital: np.nan for capital in CAPITALS}, 'composite_score': np.nan
            })
            continue
        point_rows.append({
            'action_id': action_id, 'capital_tier': ACTIONS[action_id].get('bucket'),
            'magnitude_pct': magnitude_pct, 'magnitude': magnitude,
            'status': point_status, 'reason': point_reason, **scores
        })

sweep_points_df = pd.DataFrame(point_rows)
constraint_log_df = pd.DataFrame(constraint_log)
print(f'Grid points evaluated: {len(sweep_points_df)}')
retained_mask = sweep_points_df['status'].isin(['valid', 'valid_with_skipped_action'])
print(f'Retained score points: {retained_mask.sum()} (fully applied: {(sweep_points_df["status"] == "valid").sum()} | fixed action omitted: {(sweep_points_df["status"] == "valid_with_skipped_action").sum()})')
print(f'Target grid points skipped: {(sweep_points_df["status"] == "skipped").sum()}')
if constraint_log_df.empty:
    print('No engine-enforced or library applicability constraints were triggered.')
else:
    display(constraint_log_df)

Grid points evaluated: 210
Retained score points: 204 (fully applied: 1 | fixed action omitted: 203)
Target grid points skipped: 6


,action_id,magnitude_pct,magnitude,reason,scope
0,wind_utility,0,0.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
1,wind_utility,25,250.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
2,wind_utility,50,500.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
3,wind_utility,75,750.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
4,wind_utility,100,1000.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
...,...,...,...,...,...
204,ev_charging_network,50,50.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
205,ev_charging_network,75,75.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
206,ev_charging_network,100,100.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...
207,ev_charging_network,150,150.0,"rangeland_restoration_maintenance at 56021, ma...",fixed portfolio action omitted; target point r...


In [4]:
def make_slope_rows(points):
    rows = []
    for action_id, group in points.groupby('action_id', sort=False):
        group = group.sort_values('magnitude_pct').reset_index(drop=True)
        capital_tier = group['capital_tier'].iloc[0]
        capex_per_unit = CAPEX_META[action_id]['capex_per_unit']
        for i in range(1, len(group)):
            lo = group.iloc[i - 1]
            hi = group.iloc[i]
            if lo['status'] not in {'valid', 'valid_with_skipped_action'} or hi['status'] not in {'valid', 'valid_with_skipped_action'}:
                continue
            delta_magnitude = float(hi['magnitude'] - lo['magnitude'])
            if delta_magnitude <= 0:
                continue
            delta_composite = float(hi['composite_score'] - lo['composite_score'])
            slope = delta_composite / delta_magnitude
            efficiency = slope * 1_000_000.0 / capex_per_unit if np.isfinite(capex_per_unit) and capex_per_unit > 0 else np.nan
            rows.append({
                'action_id': action_id, 'capital_tier': capital_tier,
                'magnitude_pct': float(hi['magnitude_pct']),
                'composite_score': float(hi['composite_score']),
                'E_score': float(hi['E']), 'Ec_score': float(hi['Ec']), 'S_score': float(hi['S']),
                'slope_per_unit': slope, 'cost_efficiency_per_$1M': efficiency,
                'delta_magnitude': delta_magnitude, 'lower_magnitude_pct': float(lo['magnitude_pct']),
            })
    return pd.DataFrame(rows)

marginal_df = make_slope_rows(sweep_points_df)
MARGINAL_COLUMNS = [
    'action_id', 'capital_tier', 'magnitude_pct', 'composite_score',
    'E_score', 'Ec_score', 'S_score', 'slope_per_unit',
    'cost_efficiency_per_$1M'
]
MARGINAL_PATH = DATA_DIR / 'sweep_marginal_returns.csv'
marginal_df[MARGINAL_COLUMNS].to_csv(MARGINAL_PATH, index=False)
print(f'Saved {MARGINAL_PATH}')
display(marginal_df[MARGINAL_COLUMNS].head(20))

Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/sweep_marginal_returns.csv


,action_id,capital_tier,magnitude_pct,composite_score,E_score,Ec_score,S_score,slope_per_unit,cost_efficiency_per_$1M
0,wind_utility,energy_generation,25.0,5.221011,3.330690,7.008733,5.323610,8.809051e-07,6.160176e-07
1,wind_utility,energy_generation,50.0,5.221231,3.330763,7.009284,5.323647,8.809051e-07,6.160176e-07
2,wind_utility,energy_generation,75.0,5.221451,3.330837,7.009834,5.323684,8.809051e-07,6.160176e-07
3,wind_utility,energy_generation,100.0,5.221672,3.330910,7.010385,5.323720,8.809051e-07,6.160176e-07
4,wind_utility,energy_generation,150.0,5.222112,3.331057,7.011486,5.323794,8.809051e-07,6.160176e-07
5,wind_utility,energy_generation,200.0,5.222553,3.331204,7.012587,5.323867,8.809051e-07,6.160176e-07
6,solar_utility,energy_generation,25.0,5.221647,3.330905,7.010321,5.323715,3.296538e-08,2.119687e-08
7,solar_utility,energy_generation,50.0,5.221655,3.330907,7.010342,5.323717,3.296538e-08,2.119687e-08
8,solar_utility,energy_generation,75.0,5.221663,3.330908,7.010364,5.323718,3.296538e-08,2.119687e-08
9,solar_utility,energy_generation,100.0,5.221672,3.330910,7.010385,5.323720,3.296538e-08,2.119687e-08


In [5]:
# One marginal-return curve per intervention.
for entry in PORTFOLIO:
    action_id = entry['action_id']
    group = sweep_points_df[sweep_points_df['action_id'] == action_id].sort_values('magnitude_pct')
    valid = group[group['status'].isin(['valid', 'valid_with_skipped_action'])]
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(valid['magnitude_pct'], valid['composite_score'], marker='o', linewidth=2)
    skipped = group[group['status'] == 'skipped']
    if not skipped.empty:
        ax.scatter(skipped['magnitude_pct'], [np.nan] * len(skipped), marker='x', color='crimson', label='skipped')
    ax.set_title(f'{action_id}: composite response')
    ax.set_xlabel('Magnitude (% of nominal)')
    ax.set_ylabel('Composite score')
    ax.set_xlim(0, 200)
    ax.grid(alpha=0.25)
    if not skipped.empty:
        ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'sweep_{action_id}.png', dpi=160)
    plt.close(fig)

fig, ax = plt.subplots(figsize=(13, 8))
for entry in PORTFOLIO:
    action_id = entry['action_id']
    valid = sweep_points_df[(sweep_points_df['action_id'] == action_id) & (sweep_points_df['status'].isin(['valid', 'valid_with_skipped_action']))].sort_values('magnitude_pct')
    ax.plot(valid['magnitude_pct'], valid['composite_score'], marker='o', linewidth=1.4, alpha=0.78, label=action_id)
ax.set_title('Magnitude sweep overlay — composite score')
ax.set_xlabel('Magnitude (% of nominal)')
ax.set_ylabel('Composite score')
ax.set_xlim(0, 200)
ax.grid(alpha=0.25)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7, frameon=False)
fig.tight_layout()
OVERLAY_PATH = FIG_DIR / 'sweep_overlay_composite.png'
fig.savefig(OVERLAY_PATH, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved {len(PORTFOLIO)} individual curves and {OVERLAY_PATH}')

Saved 30 individual curves and /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/sweep_overlay_composite.png


/var/folders/t3/4cm6pck10lv33361s22h2gm40000gn/T/ipykernel_52787/1755793763.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Diminishing returns and ranking

A curve is marked healthy when its consecutive composite slopes are non-increasing across the valid grid intervals. Any increase or other non-monotonic pattern is flagged for review. Rankings use the largest valid positive marginal efficiency observed across the grid, so they answer which single intervention has the strongest observed per-dollar marginal movement rather than simply which has the largest raw score change.

In [6]:
flag_rows = []
for entry in PORTFOLIO:
    action_id = entry['action_id']
    group = marginal_df[marginal_df['action_id'] == action_id].sort_values('magnitude_pct')
    slopes = group['slope_per_unit'].to_numpy()
    non_increasing = len(slopes) > 0 and bool(np.all(np.diff(slopes) <= 1e-12))
    if len(slopes) == 0:
        flag = 'review: insufficient valid intervals'
    elif non_increasing:
        flag = 'none'
    else:
        flag = 'review: increasing/non-monotonic slope'
    flag_rows.append({
        'action_id': action_id, 'capital_tier': ACTIONS[action_id].get('bucket'),
        'n_valid_intervals': len(slopes),
        'diminishing_returns': non_increasing,
        'review_flag': flag,
        'slopes_per_unit': ', '.join(f'{x:.8g}' for x in slopes),
    })
flags_df = pd.DataFrame(flag_rows)
flagged_actions_df = flags_df[flags_df['review_flag'] != 'none'].copy()
print(f'Flagged actions: {len(flagged_actions_df)}')
display(flags_df)

SOURCED_ACTIONS = {aid for aid, meta in CAPEX_META.items() if meta['usable_sourced_capex']}
UNSOURCED_ACTIONS = set(CAPEX_META) - SOURCED_ACTIONS

def _best_marginal_row(group):
    eligible = group[group['cost_efficiency_per_$1M'].notna()]
    return eligible.loc[eligible['cost_efficiency_per_$1M'].idxmax()] if not eligible.empty else None

sourced_marginal_df = marginal_df[marginal_df['action_id'].isin(SOURCED_ACTIONS)].copy()
composite_ranking = (sourced_marginal_df.groupby('action_id', as_index=False)
                     .apply(lambda g: g.loc[g['cost_efficiency_per_$1M'].idxmax()])
                     .reset_index(drop=True)
                     .sort_values('cost_efficiency_per_$1M', ascending=False))
composite_ranking = composite_ranking[['action_id', 'capital_tier', 'magnitude_pct', 'cost_efficiency_per_$1M', 'slope_per_unit']]
print(f'Usable sourced capex: {len(SOURCED_ACTIONS)} / {len(PORTFOLIO)} actions; excluded: {len(UNSOURCED_ACTIONS)}')
print('Top interventions by sourced composite points per $1M capex:')
display(composite_ranking.head(10))

ranking_rows = []
for entry in PORTFOLIO:
    action_id = entry['action_id']
    group = sweep_points_df[(sweep_points_df['action_id'] == action_id) & sweep_points_df['status'].isin(['valid', 'valid_with_skipped_action'])].sort_values('magnitude_pct')
    if len(group) >= 2 and group['magnitude_pct'].max() == 200:
        first, last = group.iloc[0], group.iloc[-1]
        raw_effects = {f'{capital}_effect_0_to_200': float(last[capital] - first[capital]) for capital in CAPITALS}
        raw_effects['composite_effect_0_to_200'] = float(last['composite_score'] - first['composite_score'])
    else:
        raw_effects = {f'{capital}_effect_0_to_200': np.nan for capital in CAPITALS}
        raw_effects['composite_effect_0_to_200'] = np.nan
    best = _best_marginal_row(marginal_df[marginal_df['action_id'] == action_id])
    if action_id in SOURCED_ACTIONS and best is not None:
        efficiency_text = f'{float(best["cost_efficiency_per_$1M"]):.12g}'
        best_pct = float(best['magnitude_pct'])
    else:
        efficiency_text = 'null — unsourced capex'
        best_pct = np.nan
    ranking_rows.append({
        'action_id': action_id, 'capital_tier': ACTIONS[action_id].get('bucket'),
        **raw_effects, 'best_magnitude_pct': best_pct,
        'capex_per_unit': CAPEX_META[action_id]['capex_per_unit'],
        'capex_source': CAPEX_META[action_id]['capex_source'] or 'null — unsourced capex',
        'cost_efficiency_per_$1M': efficiency_text,
        'cost_status': 'sourced' if action_id in SOURCED_ACTIONS else f'unsourced capex: {CAPEX_META[action_id]["capex_reason"]}',
    })
sourced_cost_ranking_df = pd.DataFrame(ranking_rows)
sourced_cost_ranking_df['_sort_efficiency'] = pd.to_numeric(sourced_cost_ranking_df['cost_efficiency_per_$1M'], errors='coerce')
sourced_cost_ranking_df = sourced_cost_ranking_df.sort_values('_sort_efficiency', ascending=False, na_position='last').drop(columns='_sort_efficiency')
SOURCED_RANKING_PATH = DATA_DIR / 'sweep_cost_ranking_sourced.csv'
sourced_cost_ranking_df.to_csv(SOURCED_RANKING_PATH, index=False)
print(f'Saved {SOURCED_RANKING_PATH}')
display(sourced_cost_ranking_df.head(10))

capital_rankings = {}
for capital, score_col in [('E', 'E_score'), ('Ec', 'Ec_score'), ('S', 'S_score')]:
    group_rows = []
    for action_id, group in sweep_points_df[sweep_points_df['status'].isin(['valid', 'valid_with_skipped_action'])].groupby('action_id', sort=False):
        if action_id not in SOURCED_ACTIONS:
            continue
        group = group.sort_values('magnitude_pct')
        intervals = []
        for i in range(1, len(group)):
            lo = group.iloc[i - 1]
            hi = group.iloc[i]
            dm = hi['magnitude'] - lo['magnitude']
            if dm <= 0:
                continue
            slope = (hi[capital] - lo[capital]) / dm
            capex = CAPEX_META[action_id]['capex_per_unit']
            efficiency = slope * 1_000_000 / capex if np.isfinite(capex) and capex > 0 else np.nan
            intervals.append((efficiency, hi['magnitude_pct'], slope))
        if intervals:
            best = max(intervals, key=lambda x: x[0])
            group_rows.append({'action_id': action_id, 'capital_tier': ACTIONS[action_id].get('bucket'), 'best_magnitude_pct': best[1], f'{capital}_cost_efficiency_per_$1M': best[0], f'{capital}_slope_per_unit': best[2]})
    capital_rankings[capital] = pd.DataFrame(group_rows).sort_values(f'{capital}_cost_efficiency_per_$1M', ascending=False)
    print(f'Top interventions by {capital} points per $1M capex:')
    display(capital_rankings[capital].head(10))

Flagged actions: 1


,action_id,capital_tier,n_valid_intervals,diminishing_returns,review_flag,slopes_per_unit
0,wind_utility,energy_generation,6,True,none,"8.8090511e-07, 8.8090511e-07, 8.8090511e-07, 8..."
1,solar_utility,energy_generation,6,True,none,"3.2965377e-08, 3.2965377e-08, 3.2965377e-08, 3..."
2,geothermal_utility,energy_generation,6,True,none,"3.0598788e-06, 3.0598788e-06, 3.0598788e-06, 3..."
3,coal_repowering,energy_generation,6,True,none,"3.274071e-07, 3.274071e-07, 3.274071e-07, 3.27..."
4,smr_advanced,energy_generation,6,True,none,"0.00013904408, 0.00013904408, 0.00013904408, 0..."
5,coal_to_solar,energy_generation,6,True,none,"2.6698972e-08, 2.6698972e-08, 2.6698972e-08, 2..."
6,battery_grid,energy_storage,6,True,none,"5.2421707e-07, 5.2421707e-07, 5.2421707e-07, 5..."
7,transmission_230kv,energy_transmission,6,True,none,"8.9831563e-05, 8.9831563e-05, 8.9831563e-05, 8..."
8,microgrid,energy_transmission,6,True,none,"0.00058790579, 0.00058790579, 0.00058790579, 0..."
9,riparian_buffer,hydrological_restoration,6,True,none,"3.0668104e-06, 3.0668104e-06, 3.0668104e-06, 3..."


Usable sourced capex: 12 / 30 actions; excluded: 18
Top interventions by sourced composite points per $1M capex:


,action_id,capital_tier,magnitude_pct,cost_efficiency_per_$1M,slope_per_unit
5,prairie_restoration,terrestrial_ecosystem,25.0,0.023851,3.577662e-06
3,invasive_treatment,terrestrial_ecosystem,100.0,0.001532,1.225311e-07
4,irrigation_efficiency,agriculture,75.0,0.001364,8.185285e-07
6,riparian_buffer,hydrological_restoration,75.0,0.000383,3.066810e-06
11,workforce_retraining,settlement_social,25.0,0.000107,9.058784e-07
9,transmission_230kv,energy_transmission,75.0,0.000036,8.983156e-05
0,affordable_housing,settlement_social,25.0,0.000018,3.289664e-06
2,health_clinic,settlement_social,25.0,0.000009,4.003275e-05
7,rural_broadband,settlement_social,50.0,0.000003,1.178733e-08
1,coal_repowering,energy_generation,50.0,0.000001,3.274071e-07


Saved /Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/sweep_cost_ranking_sourced.csv


...

,action_id,capital_tier,best_magnitude_pct,E_cost_efficiency_per_$1M,E_slope_per_unit
5,prairie_restoration,terrestrial_ecosystem,75,7.155324e-02,1.073299e-05
6,invasive_treatment,terrestrial_ecosystem,25,4.594918e-03,3.675934e-07
11,irrigation_efficiency,agriculture,25,2.557902e-03,1.534741e-06
4,riparian_buffer,hydrological_restoration,75,1.150054e-03,9.200431e-06
0,wind_utility,energy_generation,50,2.054184e-07,2.937484e-07
1,solar_utility,energy_generation,25,4.544351e-09,7.067375e-09
2,coal_repowering,energy_generation,25,0.000000e+00,0.000000e+00
3,transmission_230kv,energy_transmission,25,0.000000e+00,0.000000e+00
7,rural_broadband,settlement_social,25,0.000000e+00,0.000000e+00
8,health_clinic,settlement_social,25,0.000000e+00,0.000000e+00


Top interventions by Ec points per $1M capex:


,action_id,capital_tier,best_magnitude_pct,Ec_cost_efficiency_per_$1M,Ec_slope_per_unit
11,irrigation_efficiency,agriculture,75,1.023161e-03,6.138964e-07
3,transmission_230kv,energy_transmission,25,9.799497e-05,2.449874e-04
9,workforce_retraining,settlement_social,75,6.392429e-05,5.433565e-07
2,coal_repowering,energy_generation,150,2.045761e-06,6.137284e-07
0,wind_utility,energy_generation,200,1.539925e-06,2.202093e-06
7,rural_broadband,settlement_social,75,1.188981e-06,4.161435e-09
1,solar_utility,energy_generation,25,5.450192e-08,8.476138e-08
4,riparian_buffer,hydrological_restoration,25,0.000000e+00,0.000000e+00
5,prairie_restoration,terrestrial_ecosystem,25,0.000000e+00,0.000000e+00
6,invasive_treatment,terrestrial_ecosystem,25,0.000000e+00,0.000000e+00


Top interventions by S points per $1M capex:


,action_id,capital_tier,best_magnitude_pct,S_cost_efficiency_per_$1M,S_slope_per_unit
11,irrigation_efficiency,agriculture,75,5.115803e-04,3.069482e-07
9,workforce_retraining,settlement_social,25,2.557975e-04,2.174279e-06
10,affordable_housing,settlement_social,25,5.482773e-05,9.868991e-06
8,health_clinic,settlement_social,25,2.668850e-05,1.200983e-04
3,transmission_230kv,energy_transmission,50,9.802901e-06,2.450725e-05
7,rural_broadband,settlement_social,50,8.914447e-06,3.120056e-08
2,coal_repowering,energy_generation,25,1.228310e-06,3.684929e-07
0,wind_utility,energy_generation,75,1.027092e-07,1.468742e-07
1,solar_utility,energy_generation,50,4.544351e-09,7.067375e-09
4,riparian_buffer,hydrological_restoration,25,0.000000e+00,0.000000e+00


In [7]:
# Cross-reference the top five composite magnitude-sweep interventions with MC coefficient validation priority.
top5_ids = composite_ranking.head(5)['action_id'].tolist()
cross_rows = []
for action_id in top5_ids:
    rows = VALIDATION_DF[VALIDATION_DF['action_id'] == action_id].copy()
    high_low = bool(rows['HIGH_LEVERAGE_LOW_CONF'].fillna(False).astype(bool).any()) if not rows.empty else False
    confidences = '; '.join(sorted({str(x) for x in rows['current_confidence'].dropna() if str(x).strip()})) if not rows.empty else ''
    cross_rows.append({
        'action_id': action_id,
        'sweep_cost_efficiency_per_$1M': float(composite_ranking.loc[composite_ranking['action_id'] == action_id, 'cost_efficiency_per_$1M'].iloc[0]),
        'mc_validation_rows': int(len(rows)),
        'mc_high_leverage_low_confidence': high_low,
        'mc_current_confidence': confidences,
        'double_priority': high_low,
    })
top5_cross_reference_df = pd.DataFrame(cross_rows)
display(top5_cross_reference_df)

print('Required output files:')
print(MARGINAL_PATH)
print(SOURCED_RANKING_PATH)
print(OVERLAY_PATH)
print(f'Individual figures: {len(list(FIG_DIR.glob("sweep_*.png")))}')

,action_id,sweep_cost_efficiency_per_$1M,mc_validation_rows,mc_high_leverage_low_confidence,mc_current_confidence,double_priority
0,prairie_restoration,0.023851,1,False,high,False
1,invasive_treatment,0.001532,0,False,,False
2,irrigation_efficiency,0.001364,0,False,,False
3,riparian_buffer,0.000383,1,False,high,False
4,workforce_retraining,0.000107,0,False,,False


Required output files:
/Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/sweep_marginal_returns.csv
/Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/sweep_cost_ranking_sourced.csv
/Users/dylanhartman/projects/Energy Modeling/energy-map/data/processed/figures/sweep_overlay_composite.png
Individual figures: 31


In [ ]:
# ── Update network_metadata.json (deep-merge) ─────────────────────────────────
# F10 fix: read-modify-write so pre-existing keys are preserved.
from datetime import datetime, timezone

meta_path = DATA_DIR / 'network_metadata.json'
if meta_path.exists():
    with open(meta_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

metadata['magnitude_sweep'] = {
    'run_date':             datetime.now(timezone.utc).isoformat(),
    'notebook':             '18_magnitude_sweep.ipynb',
    'n_actions_swept':      len(PORTFOLIO),
    'n_sourced_actions':    len(SOURCED_ACTIONS),
    'marginal_csv':         'sweep_marginal_returns.csv',
    'cost_ranking_csv':     'sweep_cost_ranking_sourced.csv',
}

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Updated (deep-merge): {meta_path}")
print(f"Keys preserved: {[k for k in metadata if k != 'magnitude_sweep']}")

## Diagnostic follow-up — interpretation before reuse

The corrected sweep uses the unrounded population-weighted values from `state['county_ees']` for slope and monotonicity math; `compute_ees_summary` remains the public rounded display check. The prior quantization flags are retained in `sweep_marginal_returns_v1.csv` for comparison. The corrected flags identify only the remaining raw-shape candidates. The fixed portfolio's `rangeland_restoration_maintenance` dependency issue is separately logged in the sweep outputs.

The sourced-cost ranking excludes actions without a cited, unit-compatible per-unit capex/cost basis. Clean manufacturing has `$50,000` with `cost_unit = $/job` despite a facility magnitude; beaver reintroduction has an unsourced, unitless `$45,000` field. Prairie restoration has a sourced `$150/acre` cost proxy with matching units.

**Coefficient-MC consistency: consistent.** Prairie restoration remains near the top of the sourced-cost ranking and was the highest coefficient-sensitivity action in the MC run (`rho = 0.717`), so the two methods agree that this action deserves attention. This agreement is directional, not a substitute for validating the cost basis.